# Empirical operator-norm resource comparison

This notebook compares opt-in, non-certified empirical sizing with the existing analytical estimators. Empirical MPF values concern the repeated ideal MPF operator; aggregate-only schedule costs do not imply an implementable circuit. All plots therefore use `certification_policy="unconstrained"`.

In [ ]:
import os
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

from hamiltonian_resources import (
    BenchmarkConfig, HamiltonianSpec, MultiproductMethod, QSVTMethod,
    TimeScaling, TrotterMethod, plot_benchmark, run_benchmark,
    select_best_by_family,
)

MODEL = os.environ.get("HAMILTONIAN_EMPIRICAL_MODEL", "tfim")
REDUCED_GRID = os.environ.get("HAMILTONIAN_NOTEBOOK_REDUCED") == "1"
if MODEL == "tfim":
    model = HamiltonianSpec("transverse_field_ising", {"coupling": 1.0, "field": 3.0, "periodic": False})
elif MODEL == "heisenberg":
    model = HamiltonianSpec("heisenberg_chain", {"coupling": 1.0, "field_z": 0.3})
else:
    raise ValueError("MODEL must be 'tfim' or 'heisenberg'")

In [ ]:
system_sizes = [4, 6, 8, 12, 16, 24, 32]
target_errors = np.logspace(-1, -4, 7)
if REDUCED_GRID:
    system_sizes = [4]
    target_errors = [1e-2]

methods = [
    *(TrotterMethod(p, "empirical-operator-norm") for p in (2, 4, 6)),
    MultiproductMethod(None, error_method="empirical-operator-norm", branch_count_policy="mizuta2026-theorem6"),
    QSVTMethod(),
    *(TrotterMethod(p) for p in (2, 4, 6)),
    MultiproductMethod(None, error_method="mizuta2026-commutator-ideal-rigorous", branch_count_policy="mizuta2026-theorem6"),
]
config = BenchmarkConfig(
    hamiltonian=model, system_sizes=system_sizes, target_errors=target_errors,
    time=TimeScaling("proportional", 1.0), fixed_system_size=8,
    fixed_target_error=1e-3, methods=methods,
)
data = run_benchmark(config)
data.groupby(["sweep", "status"]).size()

In [ ]:
def best_trotter(frame, metric, sweep, policy):
    candidates = frame[(frame.method_family == "trotter") & (frame.error_policy == policy)]
    return select_best_by_family(candidates, metric, sweep=sweep, certification_policy="unconstrained")

def comparison_rows(frame, metric, sweep, comparison):
    empirical_trotter = best_trotter(frame, metric, sweep, "empirical-operator-norm").copy()
    empirical_trotter["summary_label"] = "Best empirical Trotter [non-certified]"
    empirical_mpf = frame[(frame.sweep == sweep) & (frame.method_family == "multiproduct") & (frame.error_policy == "empirical-operator-norm")].copy()
    empirical_mpf["summary_label"] = "Empirical ideal MPF [non-certified]"
    if comparison == "families":
        qsvt = frame[(frame.sweep == sweep) & (frame.method_family == "qsvt")].copy()
        qsvt["summary_label"] = "QSVT (ideal scope)"
        return pd.concat([empirical_trotter, empirical_mpf, qsvt], ignore_index=True)
    if comparison == "trotter":
        analytical = best_trotter(frame, metric, sweep, "analytical").copy()
        analytical["summary_label"] = "Best analytical Trotter"
        return pd.concat([analytical, empirical_trotter], ignore_index=True)
    refined = frame[(frame.sweep == sweep) & (frame.error_policy == "mizuta2026-commutator-ideal-rigorous")].copy()
    refined["summary_label"] = "Mizuta refined ideal MPF"
    return pd.concat([refined, empirical_mpf], ignore_index=True)

def draw_comparisons(frame, sweep):
    for metric in ("cnot_count", "t_count"):
        for comparison in ("families", "trotter", "mpf"):
            rows = comparison_rows(frame, metric, sweep, comparison)
            plot_benchmark(rows, sweep=sweep, metric=metric, series_by="summary_label", certification_policy="unconstrained")
            plt.show()

In [ ]:
draw_comparisons(data, "system-size")
draw_comparisons(data, "target-error")

In [ ]:
diagnostic_columns = [
    "sweep", "system_qubits", "target_error", "method_label",
    "trotter_order", "mpf_term_count", "segment_count",
    "empirical_calibration_id", "empirical_size_extrapolated",
    "empirical_time_extrapolated", "empirical_active_constraint",
    "mpf_exponent_sum", "mpf_exponent_sum_source",
    "mpf_explicit_schedule_available", "status", "error_message",
]
diagnostics = []
for sweep in ("system-size", "target-error"):
    for metric in ("cnot_count", "t_count"):
        selected = best_trotter(data, metric, sweep, "empirical-operator-norm").copy()
        selected["selection_metric"] = metric
        diagnostics.append(selected)
selected_trotter_diagnostics = pd.concat(diagnostics, ignore_index=True)
selected_trotter_diagnostics[["selection_metric", *diagnostic_columns[:-2]]]

In [ ]:
data[data.error_policy == "empirical-operator-norm"][diagnostic_columns]